#### 01 — Data Loading & Cleaning
- Goal: Load the raw Retail Rocket CSVs, clean them, and store in SQLite for all SQL analysis.  
- Dataset: 2.76M user events — view, addtocart, transaction

In [10]:
import pandas as pd
import sqlite3
import os
import warnings
warnings.filterwarnings('ignore')

print('Libraries imported.')

Libraries imported.


In [11]:
events = pd.read_csv(r'C:\Users\bhuvancw\OneDrive\Desktop\Data Science Projects\DA Projects\Retail Rocket - User Retention & Churn Analysis\data\events.csv')
print(f'Shape: {events.shape}')
print(f'Columns: {list(events.columns)}')
events.head()

Shape: (2756101, 5)
Columns: ['timestamp', 'visitorid', 'event', 'itemid', 'transactionid']


,timestamp,visitorid,event,itemid,transactionid
0,1433221332117,257597,view,355908,NaN
1,1433224214164,992329,view,248676,NaN
2,1433221999827,111016,view,318965,NaN
3,1433221955914,483717,view,253185,NaN
4,1433221337106,951259,view,367447,NaN


In [12]:
print("=== NULL CHECK ===")
print(events.isna().sum())
print()
print("=== EVENT TYPES ===")
print(events['event'].value_counts())

=== NULL CHECK ===
timestamp              0
visitorid              0
event                  0
itemid                 0
transactionid    2733644
dtype: int64

=== EVENT TYPES ===
event
view           2664312
addtocart        69332
transaction      22457
Name: count, dtype: int64


In [13]:
events['event_time'] = pd.to_datetime(events['timestamp'], unit = 'ms')
events['date'] = events['event_time'].dt.date
events['year_month'] = events['event_time'].dt.to_period('M').astype(str)
events['hour'] = events['event_time'].dt.hour
events['day_of_week'] = events['event_time'].dt.day_name()

events = events.rename(columns = {
    'visitorid' : 'user_id', 'event' : 'event_type',
    'itemid' : 'item_id', 'transactionid' : 'transaction_id' 
})

events.head()

,timestamp,user_id,event_type,item_id,transaction_id,event_time,date,year_month,hour,day_of_week
0,1433221332117,257597,view,355908,NaN,2015-06-02 05:02:12.117,2015-06-02,2015-06,5,Tuesday
1,1433224214164,992329,view,248676,NaN,2015-06-02 05:50:14.164,2015-06-02,2015-06,5,Tuesday
2,1433221999827,111016,view,318965,NaN,2015-06-02 05:13:19.827,2015-06-02,2015-06,5,Tuesday
3,1433221955914,483717,view,253185,NaN,2015-06-02 05:12:35.914,2015-06-02,2015-06,5,Tuesday
4,1433221337106,951259,view,367447,NaN,2015-06-02 05:02:17.106,2015-06-02,2015-06,5,Tuesday


In [16]:
before = len(events)
events = events.drop_duplicates(subset = ['user_id','event_type','item_id','timestamp'])
print(f'Removed {before - len(events):,} duplicates')
print(f'Date range: {events['event_time'].min()} → {events['event_time'].max()}')
print(f'Unique users: {events['user_id'].nunique():,}')

Removed 0 duplicates
Date range: 2015-05-03 03:00:04.384000 → 2015-09-18 02:59:47.788000
Unique users: 1,407,580


In [17]:
props1 = pd.read_csv(r'C:\Users\bhuvancw\OneDrive\Desktop\Data Science Projects\DA Projects\Retail Rocket - User Retention & Churn Analysis\data\item_properties_part1.csv')
props2 = pd.read_csv(r'C:\Users\bhuvancw\OneDrive\Desktop\Data Science Projects\DA Projects\Retail Rocket - User Retention & Churn Analysis\data\item_properties_part2.csv')

props = pd.concat([props1, props2]).drop_duplicates()
print(f'Item props shape: {props.shape}')
print(f'Properties: {props['property'].unique()}')

Item props shape: (20275902, 4)
Properties: ['categoryid' '888' '400' ... '1091' '522' '769']


In [20]:
conn = sqlite3.connect(r'C:\Users\bhuvancw\OneDrive\Desktop\Data Science Projects\DA Projects\Retail Rocket - User Retention & Churn Analysis\data\retail_rocket.db')
events.to_sql('events',conn, if_exists='replace', index = False)
props.to_sql('item_props', conn, if_exists='replace',index = False)

for t in ['events','item_props']:
    n = pd.read_sql(f'select count(*) as n from {t}', conn)['n'][0]
    print(f'{t}: {n:,} rows')

conn.close()
print('Database ready at data/retail_rocket.db')

events: 2,755,641 rows
item_props: 20,275,902 rows
Database ready at data/retail_rocket.db
